In [ ]:
#!/usr/bin/env python3
"""
Phase 0 Simulation v0.3.4

Changes vs v0.3.3:
  HW rate volatility : sigma_r = 0.015 -> 0.010  (parameter sweep v1.0, target stress 20-25%)
  Rate-OAS correlation: rho    = 0.55  -> 0.35   (parameter sweep v1.0, most conservative defensible)

Key calibrated parameters (all others unchanged from v0.3.3):
  Hull-White process : kappa=0.15, sigma_r=0.010, r0=0.045, r_bar=0.045
  OAS process        : kappa_oas=2.5, sigma_s=40 bps, oas0=70 bps, oas_bar=70 bps
  Correlation        : rho_rate_oas=0.35
  Liquidity          : liq0=0.85, liq_floor=0.05
  Simulation         : N=1000 paths, T=252 days, dt=1/252, seed=42
  Oracle staleness   : p=0.15 per path, stale_days=10

Results (summary.csv, seed=42, N=1000):
  Stress paths (min_liq < 0.20) : 22.9%
  Zero-emergency paths           : 99.6%
  Median NAV drawdown            : -4.78%
  95th pct drawdown              : -9.69%
  Oracle-emergency overlap       : 0 of 4 emergency paths

Calibration note:
  sigma_r=0.010 => ~159 bps annualized rate vol, upper range of 2022 realized vol.
  Selected via 3x3 sweep: sigma_r in {0.008, 0.010, 0.012} x rho in {0.35, 0.45, 0.55}.
  22.9% stress incidence is robust across all rho values at sigma_r=0.010.

Output directory: phase0_outputs_v03_4/
"""

from __future__ import annotations

import os
import math
import random
import argparse
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =========================
# 0) Parameter Template v0.3.1
# =========================

PARAMS: Dict = {
    "T_days": 252,
    "seed": 42,

    # ---------- State thresholds ----------
    "state_thresholds": {
        "CPR_high": 22.0,
        "CPR_low": 6.0,

        "Dur_max": 6.0,
        "WAL_max": 7.0,

        "OAS_stress": 140.0,   # bps
        "liq_min": 0.35,

        "rate_shock_thresh_daily": 0.0020,  # 20 bps/day

        # Super-state trigger:
        "emergency_min_triggers": 3,

        # ✅ NEW: Oracle freshness threshold (days)
        # If OracleAgeDays > oracle_max_age_days => ORACLE_DEGRADED
        "oracle_max_age_days": 2.0,
    },

    # ---------- Hysteresis controls ----------
    "hysteresis": {
        "min_days_in_state": 3,
        "exit_buffer": {
            "CPR": 2.0,
            "Dur": 0.25,
            "WAL": 0.25,
            "OAS": 15.0,
            "liq": 0.05,
        },
    },

    # ---------- Policy table ----------
    "policy_by_state": {
        "NORMAL": {"CF_bps": 8000, "LT_bps": 8500, "Haircut_bps": 500, "BorrowCap_mult": 1.00},
        "HIGH_PREPAY": {"CF_bps": 7500, "LT_bps": 8200, "Haircut_bps": 700, "BorrowCap_mult": 1.00},
        "EXTENSION_RISK": {"CF_bps": 6500, "LT_bps": 7200, "Haircut_bps": 1200, "BorrowCap_mult": 0.80},
        "RATE_SHOCK": {"CF_bps": 6000, "LT_bps": 6800, "Haircut_bps": 1500, "BorrowCap_mult": 0.70},
        "LIQUIDITY_STRESS": {"CF_bps": 5000, "LT_bps": 6000, "Haircut_bps": 2000, "BorrowCap_mult": 0.50},

        # Defensive postures
        "EMERGENCY_UNWIND": {"CF_bps": 0, "LT_bps": 0, "Haircut_bps": 2800, "BorrowCap_mult": 0.00},
        "ORACLE_DEGRADED": {"CF_bps": 0, "LT_bps": 6500, "Haircut_bps": 3000, "BorrowCap_mult": 0.00},
    },
    "oracle_defensive": {
    # Extra markdown applied ONLY for liquidation checks when ORACLE_DEGRADED
    # (protects against overstated stale NAV)
    "enabled": True,
    "liq_nav_markdown": 0.10,   # 10% defensive markdown
    },

    # ---------- Continuous adjustments ----------
    "continuous_adjustments": {
        "Dur_target": 4.5,
        "OAS_target": 70.0,
        "PVshock_target": 4.0,

        "CF_penalty_per_dur_year_bps": 250,
        "CF_penalty_per_oas50_bps": 150,
        "H_add_per_pvshock_pct_bps": 120,
    },

    # ---------- Vault mechanics ----------
    "vault": {
        "N_users": 250,
        "liq_bonus_bps": 500,
        "max_repay_fraction": 1.0,
        "borrow_utilization_at_t0": 0.70,
        "min_debt_to_consider": 50.0,
    },

    # ---------- NAV dynamics ----------
    "nav_model": {
        "base_carry_annual": 0.045,
        "oas_carry_beta": 0.00006,
        "price_sens_beta": 1.0,

        "convexity_beta": -25.0,
        "convexity_dynamic": True,
        "coupon_proxy": 0.055,

        "use_garch_noise": True,
        "nav_noise_daily": 0.0000,
    },

    # ---------- Reflexivity ----------
    "reflexivity": {
        "enabled": True,
        "oas_widen_bps_per_liq_ratio": 120.0,
        "liq_drop_per_liq_ratio": 0.30,
        "liq_floor": 0.05,
        "oas_cap": 500.0,
    },

    # ---------- Oracle staleness (deterministic scenarios) ----------
    "oracle_staleness": {
        "enabled_for_scenarios": ["F_oracle_stale_gap"],
        "stale_start_day": 60,
        "stale_days": 10,
        "freeze_fields": ["NAV", "CPR", "Dur", "WAL", "PV_up100_pct", "PV_dn100_pct"],
    },

    # ---------- Oracle staleness (stochastic) ----------
    "oracle_staleness_stochastic": {
        "enabled": True,
        "path_probability": 0.15,
        "stale_start_day": 60,
        "stale_days": 10,
        "freeze_fields": ["NAV", "CPR", "Dur", "WAL", "PV_up100_pct", "PV_dn100_pct"],
    },

    # ---------- Risk mappings ----------
    "risk_mappings": {
        "cpr_max": 28.0,
        "cpr_min": 4.0,
        "cpr_rate_mid": 0.045,
        "cpr_rate_slope": 110.0,
        "cpr_oas_penalty_per_100bp": 2.0,

        "dur_base": 4.5,
        "dur_cpr_slope": 0.12,
        "dur_rate_slope": 20.0,
        "dur_clip": (2.5, 8.5),

        "wal_base": 5.0,
        "wal_cpr_slope": 0.10,
        "wal_clip": (3.0, 10.0),

        "pvshock_clip": (0.5, 12.0),
    },

    # ---------- Stochastic drivers ----------
    "stochastic": {
        "dt": 1/252,

        # ✅ NEW: default paths = 1000
        "n_paths_default": 1000,

        "hw": {"r0": 0.045, "a": 0.15, "sigma": 0.010, "r_bar": 0.045},
        "oas": {"oas0": 70.0, "kappa": 2.5, "sigma": 40.0, "oas_bar": 70.0},
        "corr": {"rho_rate_oas": 0.35},
        "liquidity": {
            "liq0": 0.85, "liq_floor": 0.05, "liq_ceiling": 0.95,
            "oas_to_liq_slope": 0.0025, "oas_ref": 70.0
        },
        "garch": {"enabled": True, "omega": 1e-7, "alpha": 0.08, "beta": 0.90, "eps0": 0.0, "h0": 1e-6},
    },
}


# =========================
# Helpers
# =========================

def clamp(x: float, lo: float, hi: float) -> float:
    return max(lo, min(hi, x))

def logistic(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


# =========================
# 1) Deterministic Scenarios
# =========================

def scenario_base(T: int) -> pd.DataFrame:
    return pd.DataFrame({"rate": np.full(T, 0.045), "oas": np.full(T, 70.0), "liq": np.full(T, 0.85)})

def scenario_rate_up_150bp(T: int) -> pd.DataFrame:
    rate = np.full(T, 0.045)
    ramp_days = min(30, T - 1)
    rate[:ramp_days] = np.linspace(0.045, 0.060, ramp_days)
    rate[ramp_days:] = rate[ramp_days - 1]
    oas = np.full(T, 70.0)
    oas[:ramp_days] += np.linspace(0.0, 30.0, ramp_days)
    liq = np.full(T, 0.80)
    return pd.DataFrame({"rate": rate, "oas": oas, "liq": liq})

def scenario_rate_down_150bp(T: int) -> pd.DataFrame:
    rate = np.full(T, 0.045)
    ramp_days = min(30, T - 1)
    rate[:ramp_days] = np.linspace(0.045, 0.030, ramp_days)
    rate[ramp_days:] = rate[ramp_days - 1]
    oas = np.full(T, 70.0)
    oas[:ramp_days] += np.linspace(0.0, -20.0, ramp_days)
    liq = np.full(T, 0.88)
    return pd.DataFrame({"rate": rate, "oas": oas, "liq": liq})

def scenario_liquidity_air_pocket(T: int) -> pd.DataFrame:
    rate = np.full(T, 0.045)
    oas = np.full(T, 70.0)
    liq = np.full(T, 0.85)
    shock_start, shock_len = 40, 20
    if shock_start + shock_len < T:
        oas[shock_start:shock_start + shock_len] = 190.0
        liq[shock_start:shock_start + shock_len] = 0.25
    return pd.DataFrame({"rate": rate, "oas": oas, "liq": liq})

def scenario_whipsaw(T: int) -> pd.DataFrame:
    rate = np.full(T, 0.045)
    up_days, down_days = 20, 20
    if up_days + down_days < T:
        rate[:up_days] = np.linspace(0.045, 0.055, up_days)
        rate[up_days:up_days + down_days] = np.linspace(0.055, 0.045, down_days)
        rate[up_days + down_days:] = 0.045
    oas = np.full(T, 70.0)
    oas[:up_days + down_days] += np.linspace(0, 40, up_days + down_days)
    liq = np.full(T, 0.75)
    return pd.DataFrame({"rate": rate, "oas": oas, "liq": liq})

def scenario_oracle_stale_gap(T: int) -> pd.DataFrame:
    rate = np.full(T, 0.045)
    if T > 80:
        rate[55:80] = np.linspace(0.045, 0.053, 25)
        rate[80:] = rate[79]
    oas = np.full(T, 70.0)
    oas[55:80] += np.linspace(0, 50, 25)
    liq = np.full(T, 0.85)
    return pd.DataFrame({"rate": rate, "oas": oas, "liq": liq})

SCENARIOS = {
    "A_base": scenario_base,
    "B_rate_up_150bp": scenario_rate_up_150bp,
    "C_rate_down_150bp": scenario_rate_down_150bp,
    "D_liq_air_pocket": scenario_liquidity_air_pocket,
    "E_whipsaw": scenario_whipsaw,
    "F_oracle_stale_gap": scenario_oracle_stale_gap,
}


# =========================
# 2) Stochastic Drivers
# =========================

def correlated_normals(T: int, rho: float, seed: int) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    z1 = rng.standard_normal(T)
    z2 = rng.standard_normal(T)
    e1 = z1
    e2 = rho * z1 + np.sqrt(max(1e-12, 1 - rho**2)) * z2
    return e1, e2

def generate_hw_rates(T: int, dt: float, r0: float, a: float, sigma: float, r_bar: float, z: np.ndarray) -> np.ndarray:
    r = np.zeros(T)
    r[0] = r0
    for t in range(1, T):
        dr = a * (r_bar - r[t-1]) * dt + sigma * np.sqrt(dt) * z[t]
        r[t] = r[t-1] + dr
    return r

def generate_oas(T: int, dt: float, oas0: float, kappa: float, sigma_bps: float, oas_bar: float, z: np.ndarray) -> np.ndarray:
    o = np.zeros(T)
    o[0] = oas0
    for t in range(1, T):
        do = kappa * (oas_bar - o[t-1]) * dt + sigma_bps * np.sqrt(dt) * z[t]
        o[t] = max(0.0, o[t-1] + do)
    return o

def liquidity_from_oas(oas_bps: np.ndarray, liq0: float, slope: float, floor: float, ceiling: float, oas_ref: float) -> np.ndarray:
    liq = liq0 - slope * (oas_bps - oas_ref)
    return np.clip(liq, floor, ceiling)

def generate_stochastic_drivers(T: int, p: Dict, seed: int) -> pd.DataFrame:
    st = p["stochastic"]
    dt = st["dt"]

    rho = st["corr"]["rho_rate_oas"]
    z_rate, z_oas = correlated_normals(T, rho, seed)

    hw = st["hw"]
    rates = generate_hw_rates(T, dt, hw["r0"], hw["a"], hw["sigma"], hw["r_bar"], z_rate)

    oasp = st["oas"]
    oas = generate_oas(T, dt, oasp["oas0"], oasp["kappa"], oasp["sigma"], oasp["oas_bar"], z_oas)

    lp = st["liquidity"]
    liq = liquidity_from_oas(oas, lp["liq0"], lp["oas_to_liq_slope"], lp["liq_floor"], lp["liq_ceiling"], lp["oas_ref"])

    return pd.DataFrame({"rate": rates, "oas": oas, "liq": liq})


# =========================
# 3) GARCH(1,1)
# =========================

def garch11(T: int, omega: float, alpha: float, beta: float, eps0: float, h0: float, seed: int) -> np.ndarray:
    rng = np.random.default_rng(seed)
    eps = np.zeros(T)
    h = np.zeros(T)
    eps[0] = eps0
    h[0] = h0
    for t in range(1, T):
        z = rng.standard_normal()
        h[t] = omega + alpha * (eps[t-1] ** 2) + beta * h[t-1]
        eps[t] = np.sqrt(max(h[t], 1e-18)) * z
    return eps


# =========================
# 4) Risk Vector Generation
# =========================

def compute_cpr(rate: float, oas_bps: float, p: Dict) -> float:
    m = p["risk_mappings"]
    z = -(rate - m["cpr_rate_mid"]) * m["cpr_rate_slope"]
    raw = logistic(z)
    cpr = m["cpr_min"] + (m["cpr_max"] - m["cpr_min"]) * raw
    cpr -= (oas_bps / 100.0) * m["cpr_oas_penalty_per_100bp"]
    return clamp(cpr, m["cpr_min"], m["cpr_max"])

def compute_duration(rate: float, cpr: float, p: Dict) -> float:
    m = p["risk_mappings"]
    cpr_baseline = 12.0
    dur = m["dur_base"]
    dur += m["dur_rate_slope"] * (rate - 0.045)
    dur += m["dur_cpr_slope"] * max(0.0, (cpr_baseline - cpr))
    return clamp(dur, m["dur_clip"][0], m["dur_clip"][1])

def compute_wal(cpr: float, dur: float, p: Dict) -> float:
    m = p["risk_mappings"]
    cpr_baseline = 12.0
    wal = m["wal_base"]
    wal += 0.40 * (dur - m["dur_base"])
    wal += m["wal_cpr_slope"] * max(0.0, (cpr_baseline - cpr))
    return clamp(wal, m["wal_clip"][0], m["wal_clip"][1])

def compute_pv_shocks(duration: float, p: Dict) -> Tuple[float, float]:
    m = p["risk_mappings"]
    pv = clamp(duration, m["pvshock_clip"][0], m["pvshock_clip"][1])
    return pv, pv

def convexity_beta_dynamic(rate: float, coupon_proxy: float, base_beta: float) -> float:
    m = rate - coupon_proxy
    scale = math.exp(- (m / 0.01) ** 2)  # width ~100bp
    return base_beta * (0.5 + 0.8 * scale)

def generate_risk_vector(drivers: pd.DataFrame, p: Dict, noise_series: Optional[np.ndarray] = None) -> pd.DataFrame:
    T = len(drivers)
    nav = np.zeros(T)
    cpr = np.zeros(T)
    dur = np.zeros(T)
    wal = np.zeros(T)
    pv_up = np.zeros(T)
    pv_dn = np.zeros(T)

    nav[0] = 100.0
    nm = p["nav_model"]

    for t in range(T):
        rate_t = float(drivers.loc[t, "rate"])
        oas_t = float(drivers.loc[t, "oas"])

        cpr[t] = compute_cpr(rate_t, oas_t, p)
        dur[t] = compute_duration(rate_t, cpr[t], p)
        wal[t] = compute_wal(cpr[t], dur[t], p)
        pv_up[t], pv_dn[t] = compute_pv_shocks(dur[t], p)

        if t < T - 1:
            carry_annual = nm["base_carry_annual"] + nm["oas_carry_beta"] * oas_t
            carry_daily = carry_annual / 365.0

            dr = float(drivers.loc[t + 1, "rate"] - drivers.loc[t, "rate"])
            price_linear = -nm["price_sens_beta"] * dur[t] * dr

            cb = nm["convexity_beta"]
            if nm.get("convexity_dynamic", False):
                cb = convexity_beta_dynamic(rate_t, nm.get("coupon_proxy", 0.055), nm["convexity_beta"])
            price_convex = 0.5 * cb * (dr ** 2)

            noise = float(noise_series[t]) if noise_series is not None else np.random.normal(0, nm["nav_noise_daily"])
            nav[t + 1] = nav[t] * (1.0 + carry_daily + price_linear + price_convex + noise)

    out = drivers.copy()
    out["NAV"] = nav
    out["CPR"] = cpr
    out["Dur"] = dur
    out["WAL"] = wal
    out["PV_up100_pct"] = pv_up
    out["PV_dn100_pct"] = pv_dn
    out["dRate"] = out["rate"].diff().fillna(0.0)

    # ✅ NEW: oracle age exists in all paths (0 by default)
    out["OracleAgeDays"] = 0.0
    return out


# =========================
# 5) Oracle staleness (Option A)
# =========================

def apply_oracle_staleness(
    risk_df: pd.DataFrame,
    freeze_fields: List[str],
    stale_start_day: int,
    stale_days: int,
    drivers: pd.DataFrame,
    p: Dict,
    noise_series: Optional[np.ndarray],
) -> pd.DataFrame:
    """
    Freeze oracle-published fields during a stale window.
    Track OracleAgeDays, and after stale window splice "true" recomputed values (gap).
    """
    risk = risk_df.copy()
    T = len(risk)
    s0 = int(stale_start_day)
    n = int(stale_days)
    if s0 <= 0 or s0 >= T:
        return risk

    s1 = min(s0 + n, T - 1)

    # Freeze published values to last good print
    for col in freeze_fields:
        last_good = risk.at[s0 - 1, col]
        risk.loc[s0:s1, col] = last_good

    # ✅ NEW: OracleAgeDays tracking
    # age=0 before staleness; increments each day during stale; resets after wake.
    if "OracleAgeDays" not in risk.columns:
        risk["OracleAgeDays"] = 0.0
    risk.loc[s0:s1, "OracleAgeDays"] = np.arange(1, (s1 - s0) + 2, dtype=float)
    if s1 + 1 < T:
        risk.loc[s1 + 1:, "OracleAgeDays"] = 0.0

    # Wake up: recompute true values and splice post-stale
    if s1 + 1 < T:
        true_risk = generate_risk_vector(drivers, p, noise_series=noise_series)
        for col in freeze_fields:
            risk.loc[s1 + 1:, col] = true_risk.loc[s1 + 1:, col]

    # Recompute dRate since rate driver moves regardless of freeze
    risk["dRate"] = risk["rate"].diff().fillna(0.0)
    return risk


# =========================
# 6) State Machine (uses OracleAgeDays)
# =========================

def classify_state_row(row: pd.Series, p: Dict) -> str:
    th = p["state_thresholds"]

    # ✅ NEW: oracle freshness check
    max_age = th.get("oracle_max_age_days", None)
    if max_age is not None:
        age = float(row.get("OracleAgeDays", 0.0))
        if age > float(max_age):
            return "ORACLE_DEGRADED"

    required = ["NAV", "CPR", "Dur", "WAL", "oas", "liq", "dRate", "PV_up100_pct"]
    if any(pd.isna(row.get(k)) for k in required):
        return "ORACLE_DEGRADED"

    flags = {
        "low_cpr": float(row["CPR"]) < th["CPR_low"],
        "high_dur": float(row["Dur"]) > th["Dur_max"],
        "high_wal": float(row["WAL"]) > th["WAL_max"],
        "high_oas": float(row["oas"]) > th["OAS_stress"],
        "low_liq": float(row["liq"]) < th["liq_min"],
        "rate_shock": abs(float(row["dRate"])) > th["rate_shock_thresh_daily"],
    }
    stress_count = sum(1 for v in flags.values() if v)

    if stress_count >= th["emergency_min_triggers"]:
        return "EMERGENCY_UNWIND"
    if flags["high_oas"] or flags["low_liq"]:
        return "LIQUIDITY_STRESS"
    if flags["rate_shock"]:
        return "RATE_SHOCK"
    if flags["high_dur"] or flags["high_wal"] or flags["low_cpr"]:
        return "EXTENSION_RISK"
    if float(row["CPR"]) > th["CPR_high"]:
        return "HIGH_PREPAY"
    return "NORMAL"

def apply_hysteresis(states_raw: List[str], risk_df: pd.DataFrame, p: Dict) -> List[str]:
    h = p["hysteresis"]
    th = p["state_thresholds"]
    eb = h["exit_buffer"]

    out: List[str] = []
    current = states_raw[0]
    days_in = 0

    def can_exit(prev_state: str, t: int, days_in_state: int) -> bool:
        row = risk_df.iloc[t]

        if prev_state in ("ORACLE_DEGRADED", "EMERGENCY_UNWIND"):
            if prev_state == "ORACLE_DEGRADED":
                return states_raw[t] != "ORACLE_DEGRADED"
            return (days_in_state >= h["min_days_in_state"]) and (states_raw[t] != "EMERGENCY_UNWIND")

        if days_in_state < h["min_days_in_state"]:
            return False

        if prev_state == "HIGH_PREPAY":
            return float(row["CPR"]) < (th["CPR_high"] - eb["CPR"])
        if prev_state == "EXTENSION_RISK":
            return (float(row["CPR"]) > (th["CPR_low"] + eb["CPR"])
                    and float(row["Dur"]) < (th["Dur_max"] - eb["Dur"])
                    and float(row["WAL"]) < (th["WAL_max"] - eb["WAL"]))
        if prev_state == "LIQUIDITY_STRESS":
            return (float(row["oas"]) < (th["OAS_stress"] - eb["OAS"])
                    and float(row["liq"]) > (th["liq_min"] + eb["liq"]))
        if prev_state == "RATE_SHOCK":
            return abs(float(row["dRate"])) < (0.5 * th["rate_shock_thresh_daily"])
        return True

    for t in range(len(states_raw)):
        raw = states_raw[t]
        if t == 0:
            out.append(current)
            days_in = 1
            continue

        if raw == current:
            out.append(current)
            days_in += 1
            continue

        if can_exit(current, t, days_in):
            current = raw
            days_in = 1

        out.append(current)
        days_in += 1

    return out


# =========================
# 7) Policy Engine
# =========================

def compute_policy(row: pd.Series, state: str, p: Dict) -> Dict[str, float]:
    base = p["policy_by_state"][state]
    adj = p["continuous_adjustments"]

    CF = float(base["CF_bps"])
    LT = float(base["LT_bps"])
    H = float(base["Haircut_bps"])
    cap_mult = float(base["BorrowCap_mult"])

    if state in ("ORACLE_DEGRADED", "EMERGENCY_UNWIND"):
        return {"CF_bps": CF, "LT_bps": LT, "Haircut_bps": H, "BorrowCap_mult": cap_mult}

    dur_excess = max(0.0, float(row["Dur"]) - adj["Dur_target"])
    oas_excess_50 = max(0.0, (float(row["oas"]) - adj["OAS_target"]) / 50.0)
    pv_excess = max(0.0, float(row["PV_up100_pct"]) - adj["PVshock_target"])

    CF -= adj["CF_penalty_per_dur_year_bps"] * dur_excess
    CF -= adj["CF_penalty_per_oas50_bps"] * oas_excess_50
    H += adj["H_add_per_pvshock_pct_bps"] * pv_excess

    CF = clamp(CF, 0.0, 9000.0)
    LT = clamp(LT, 0.0, 9500.0)
    H = clamp(H, 0.0, 4000.0)

    if LT > 0 and CF > 0:
        LT = max(LT, CF + 300)

    return {"CF_bps": CF, "LT_bps": LT, "Haircut_bps": H, "BorrowCap_mult": cap_mult}


# =========================
# 8) Vault Simulation (+ reflexivity)
# =========================

@dataclass
class User:
    shares: float
    debt: float
    liquidated: bool = False

def init_users(nav0: float, p: Dict) -> List[User]:
    v = p["vault"]
    N = v["N_users"]
    rng = np.random.default_rng(p["seed"])

    values = rng.lognormal(mean=9.2, sigma=0.9, size=N)
    values = np.clip(values, 5_000, 10_000_000)
    shares = values / nav0

    CF0 = p["policy_by_state"]["NORMAL"]["CF_bps"] / 10000.0
    util = v["borrow_utilization_at_t0"]
    debt = values * CF0 * util

    return [User(float(shares[i]), float(debt[i])) for i in range(N)]

def simulate_vault_with_reflexivity(risk_df: pd.DataFrame, p: Dict) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    v = p["vault"]
    ref = p["reflexivity"]
    users = init_users(nav0=float(risk_df.loc[0, "NAV"]), p=p)

    liq_events = []
    daily_rows = []
    df = risk_df.copy()

    for t in range(len(df)):
        row = df.iloc[t]
        NAV = float(row["NAV"])
        state = str(row["State"])
        LT = float(row["LT_bps"]) / 10000.0
        haircut = float(row["Haircut_bps"]) / 10000.0
        # --- Defensive oracle posture: use a conservative NAV for liquidation checks ---
        NAV_liq = NAV
        od = p.get("oracle_defensive", {})
        if od.get("enabled", True) and state == "ORACLE_DEGRADED":
            NAV_liq = NAV * (1.0 - float(od.get("liq_nav_markdown", 0.10)))
        liq_bonus = v["liq_bonus_bps"] / 10000.0

        total_debt = 0.0
        total_collateral_value = 0.0
        liquidations_today = 0
        liq_volume_today = 0.0

        for i, u in enumerate(users):
            if u.liquidated or u.debt < v["min_debt_to_consider"]:
                continue

            coll_value = u.shares * NAV
            coll_value_liq = u.shares * NAV_liq
            eff_coll_value = coll_value_liq * (1.0 - haircut)

            total_debt += u.debt
            total_collateral_value += coll_value

            if LT > 0 and u.debt > eff_coll_value * LT:
                repay = u.debt * v["max_repay_fraction"]
                seize_value = repay * (1.0 + liq_bonus)
                seize_shares = min(seize_value / NAV, u.shares)
                repay = min(repay, u.debt)

                u.shares -= seize_shares
                u.debt -= repay

                liquidations_today += 1
                liq_volume_today += repay

                liq_events.append({
                    "day": t, "user_id": i, "state": state,
                    "NAV": NAV, "haircut": haircut, "LT": LT,
                    "repay": repay, "seize_shares": seize_shares,
                    "remaining_debt": u.debt, "remaining_shares": u.shares
                })

                if u.debt < v["min_debt_to_consider"] or u.shares <= 1e-9:
                    u.liquidated = True

        hfs = []
        for u in users:
            if u.liquidated or u.debt < v["min_debt_to_consider"]:
                continue
            eff_coll = (u.shares * NAV) * (1.0 - haircut)
            if u.debt > 0 and LT > 0:
                hfs.append((eff_coll * LT) / u.debt)

        daily_rows.append({
            "day": t, "State": state,
            "NAV": NAV,
            "rate": float(row["rate"]),
            "oas": float(row["oas"]),
            "liq": float(row["liq"]),
            "CF": float(row["CF_bps"]) / 10000.0,
            "LT": LT,
            "Haircut": haircut,
            "total_debt": total_debt,
            "total_collateral_value": total_collateral_value,
            "liquidations": liquidations_today,
            "liq_volume": liq_volume_today,
            "avg_HF": float(np.mean(hfs)) if hfs else np.nan,
            "p10_HF": float(np.quantile(hfs, 0.10)) if len(hfs) >= 5 else np.nan,
            "p50_HF": float(np.quantile(hfs, 0.50)) if len(hfs) >= 5 else np.nan,
        })

        if ref.get("enabled", True) and t < len(df) - 1 and total_collateral_value > 0:
            liq_ratio = liq_volume_today / total_collateral_value
            widen = ref["oas_widen_bps_per_liq_ratio"] * (liq_ratio * 100.0)
            df.at[t + 1, "oas"] = min(ref["oas_cap"], float(df.at[t + 1, "oas"]) + widen)

            liq_drop = ref["liq_drop_per_liq_ratio"] * (liq_ratio * 100.0)
            df.at[t + 1, "liq"] = max(ref["liq_floor"], float(df.at[t + 1, "liq"]) - liq_drop)

    return pd.DataFrame(daily_rows), pd.DataFrame(liq_events), df


# =========================
# 9) Policy surface plots
# =========================

def risk_surface_plot(p: Dict, outdir: str) -> None:
    ensure_dir(outdir)

    d_rates = np.linspace(-0.004, 0.004, 81)
    oas_vals = np.linspace(40, 250, 81)

    base_rate = 0.045
    base_liq = 0.80
    NAV = 100.0

    Z_h = np.zeros((len(oas_vals), len(d_rates)))
    Z_cf = np.zeros((len(oas_vals), len(d_rates)))

    for yi, oas in enumerate(oas_vals):
        for xi, dr in enumerate(d_rates):
            cpr = compute_cpr(base_rate, oas, p)
            dur = compute_duration(base_rate, cpr, p)
            wal = compute_wal(cpr, dur, p)
            pv_up, pv_dn = compute_pv_shocks(dur, p)

            row = pd.Series({
                "NAV": NAV,
                "rate": base_rate,
                "oas": oas,
                "liq": base_liq,
                "CPR": cpr,
                "Dur": dur,
                "WAL": wal,
                "PV_up100_pct": pv_up,
                "PV_dn100_pct": pv_dn,
                "dRate": dr,
                "OracleAgeDays": 0.0,
            })

            state = classify_state_row(row, p)
            pol = compute_policy(row, state, p)
            Z_h[yi, xi] = pol["Haircut_bps"] / 10000.0
            Z_cf[yi, xi] = pol["CF_bps"] / 10000.0

    fig = plt.figure(figsize=(9, 6))
    plt.imshow(Z_h, origin="lower", aspect="auto",
               extent=[d_rates.min(), d_rates.max(), oas_vals.min(), oas_vals.max()])
    plt.colorbar(label="Haircut")
    plt.xlabel("Daily Δr")
    plt.ylabel("OAS (bps)")
    plt.title("Policy Surface: Haircut as function of Δr and OAS")
    plt.tight_layout()
    fig.savefig(os.path.join(outdir, "PolicySurface_Haircut.png"))
    plt.close(fig)

    fig = plt.figure(figsize=(9, 6))
    plt.imshow(Z_cf, origin="lower", aspect="auto",
               extent=[d_rates.min(), d_rates.max(), oas_vals.min(), oas_vals.max()])
    plt.colorbar(label="Collateral Factor (CF)")
    plt.xlabel("Daily Δr")
    plt.ylabel("OAS (bps)")
    plt.title("Policy Surface: CF as function of Δr and OAS")
    plt.tight_layout()
    fig.savefig(os.path.join(outdir, "PolicySurface_CF.png"))
    plt.close(fig)


# =========================
# 10) Pipeline runner
# =========================

def run_pipeline(drivers: pd.DataFrame, p: Dict,
                 oracle_stale: Optional[Dict] = None,
                 garch_seed: Optional[int] = None) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    T = len(drivers)

    noise_series = None
    nm = p["nav_model"]
    stoch = p["stochastic"]
    if nm.get("use_garch_noise", False) and stoch["garch"]["enabled"] and garch_seed is not None:
        g = stoch["garch"]
        noise_series = garch11(T, g["omega"], g["alpha"], g["beta"], g["eps0"], g["h0"], seed=garch_seed)

    risk = generate_risk_vector(drivers, p, noise_series=noise_series)

    if oracle_stale is not None and oracle_stale.get("enabled", True):
        risk = apply_oracle_staleness(
            risk_df=risk,
            freeze_fields=oracle_stale["freeze_fields"],
            stale_start_day=oracle_stale["stale_start_day"],
            stale_days=oracle_stale["stale_days"],
            drivers=drivers,
            p=p,
            noise_series=noise_series
        )

    states_raw = [classify_state_row(risk.iloc[t], p) for t in range(len(risk))]
    states = apply_hysteresis(states_raw, risk, p)
    risk["State_raw"] = states_raw
    risk["State"] = states

    pol = [compute_policy(risk.iloc[t], str(risk.iloc[t]["State"]), p) for t in range(len(risk))]
    pol_df = pd.DataFrame(pol)
    risk = pd.concat([risk.reset_index(drop=True), pol_df.reset_index(drop=True)], axis=1)

    daily1, liqs1, risk_reflex_drivers = simulate_vault_with_reflexivity(risk, p)

    drivers2 = risk_reflex_drivers[["rate", "oas", "liq"]].copy()
    risk2 = generate_risk_vector(drivers2, p, noise_series=noise_series)

    if oracle_stale is not None and oracle_stale.get("enabled", True):
        risk2 = apply_oracle_staleness(
            risk_df=risk2,
            freeze_fields=oracle_stale["freeze_fields"],
            stale_start_day=oracle_stale["stale_start_day"],
            stale_days=oracle_stale["stale_days"],
            drivers=drivers2,
            p=p,
            noise_series=noise_series
        )

    states_raw2 = [classify_state_row(risk2.iloc[t], p) for t in range(len(risk2))]
    states2 = apply_hysteresis(states_raw2, risk2, p)
    risk2["State_raw"] = states_raw2
    risk2["State"] = states2

    pol2 = [compute_policy(risk2.iloc[t], str(risk2.iloc[t]["State"]), p) for t in range(len(risk2))]
    pol2_df = pd.DataFrame(pol2)
    risk2 = pd.concat([risk2.reset_index(drop=True), pol2_df.reset_index(drop=True)], axis=1)

    daily, liqs, _ = simulate_vault_with_reflexivity(risk2, p)
    return risk2, daily, liqs


# =========================
# 11) Scenario + Stochastic runners
# =========================

def save_line(y, title, ylabel, outpath):
    fig = plt.figure()
    plt.plot(y)
    plt.title(title)
    plt.xlabel("Day")
    plt.ylabel(ylabel)
    plt.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)

def plot_bundle(risk: pd.DataFrame, daily: pd.DataFrame, outdir: str, prefix: str) -> None:
    ensure_dir(outdir)
    save_line(risk["rate"].values, f"{prefix} - Rate", "Rate", os.path.join(outdir, f"{prefix}_rate.png"))
    save_line(risk["oas"].values, f"{prefix} - OAS (bps)", "OAS (bps)", os.path.join(outdir, f"{prefix}_oas.png"))
    save_line(risk["liq"].values, f"{prefix} - Liquidity", "Liquidity", os.path.join(outdir, f"{prefix}_liq.png"))
    save_line(risk["NAV"].values, f"{prefix} - NAV", "NAV", os.path.join(outdir, f"{prefix}_nav.png"))
    save_line(risk["Dur"].values, f"{prefix} - Duration", "Years", os.path.join(outdir, f"{prefix}_dur.png"))
    save_line(risk["CPR"].values, f"{prefix} - CPR", "CPR (%)", os.path.join(outdir, f"{prefix}_cpr.png"))
    save_line((risk["CF_bps"]/10000.0).values, f"{prefix} - CF", "CF", os.path.join(outdir, f"{prefix}_CF.png"))
    save_line((risk["Haircut_bps"]/10000.0).values, f"{prefix} - Haircut", "Haircut", os.path.join(outdir, f"{prefix}_Haircut.png"))
    save_line(daily["liquidations"].values, f"{prefix} - Liquidations/day", "#", os.path.join(outdir, f"{prefix}_liquidations.png"))

def run_scenarios(p: Dict, outdir: str) -> None:
    sdir = os.path.join(outdir, "scenarios")
    ensure_dir(sdir)

    for name, fn in SCENARIOS.items():
        print(f"[scenarios] running {name} ...")
        T = p["T_days"]
        drivers = fn(T)

        oracle_stale = None
        if name in p["oracle_staleness"]["enabled_for_scenarios"]:
            oracle_stale = {
                "enabled": True,
                "stale_start_day": p["oracle_staleness"]["stale_start_day"],
                "stale_days": p["oracle_staleness"]["stale_days"],
                "freeze_fields": p["oracle_staleness"]["freeze_fields"],
            }

        risk, daily, liqs = run_pipeline(drivers, p, oracle_stale=oracle_stale, garch_seed=None)

        risk.to_csv(os.path.join(sdir, f"{name}_risk_policy.csv"), index=False)
        daily.to_csv(os.path.join(sdir, f"{name}_vault_daily.csv"), index=False)
        liqs.to_csv(os.path.join(sdir, f"{name}_liquidations.csv"), index=False)

        plot_bundle(risk, daily, sdir, name)

    risk_surface_plot(p, outdir)
    print(f"[scenarios] done. outputs in {sdir}/ and policy surfaces in {outdir}/")

def histogram(series: np.ndarray, title: str, xlabel: str, outpath: str) -> None:
    fig = plt.figure()
    plt.hist(series, bins=30)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    fig.savefig(outpath)
    plt.close(fig)

def run_stochastic(p: Dict, outdir: str, n_paths: int, save_sample_paths: int = 5) -> None:
    sdir = os.path.join(outdir, "stochastic")
    ensure_dir(sdir)
    sample_dir = os.path.join(sdir, "path_samples")
    ensure_dir(sample_dir)

    T = p["T_days"]
    ocs = p["oracle_staleness_stochastic"]
    rng = np.random.default_rng(p["seed"])

    stats_rows = []
    max_liqs = []
    max_liq_vols = []
    max_oas = []
    min_liq = []
    max_dd = []
    emergency_days = []
    oracle_gap_flags = []

    for k in range(n_paths):
        seed_k = int(p["seed"] + 10_000 + k)
        drivers = generate_stochastic_drivers(T, p, seed=seed_k)

        oracle_cfg = None
        gap_flag = False
        if ocs.get("enabled", True) and (rng.uniform() < float(ocs["path_probability"])):
            gap_flag = True
            oracle_cfg = {
                "enabled": True,
                "stale_start_day": ocs["stale_start_day"],
                "stale_days": ocs["stale_days"],
                "freeze_fields": ocs["freeze_fields"],
            }

        garch_seed = seed_k + 12345 if (p["nav_model"].get("use_garch_noise", False) and p["stochastic"]["garch"]["enabled"]) else None
        risk, daily, liqs = run_pipeline(drivers, p, oracle_stale=oracle_cfg, garch_seed=garch_seed)

        max_liq = float(np.nanmax(daily["liquidations"].values)) if len(daily) else 0.0
        max_liq_vol = float(np.nanmax(daily["liq_volume"].values)) if len(daily) else 0.0
        max_o = float(np.nanmax(risk["oas"].values))
        min_l = float(np.nanmin(risk["liq"].values))

        nav = risk["NAV"].values
        peak = np.maximum.accumulate(nav)
        drawdown = (nav / np.maximum(peak, 1e-12)) - 1.0
        max_drawdown = float(np.nanmin(drawdown))

        em_days = int(np.sum(risk["State"].values == "EMERGENCY_UNWIND"))

        stats = {
            "path": k,
            "max_daily_liquidations": max_liq,
            "max_daily_liq_volume": max_liq_vol,
            "max_oas_bps": max_o,
            "min_liquidity": min_l,
            "max_drawdown": max_drawdown,
            "emergency_days": em_days,
            "oracle_stale_gap": int(gap_flag),
        }
        stats_rows.append(stats)

        max_liqs.append(max_liq)
        max_liq_vols.append(max_liq_vol)
        max_oas.append(max_o)
        min_liq.append(min_l)
        max_dd.append(max_drawdown)
        emergency_days.append(em_days)
        oracle_gap_flags.append(int(gap_flag))

        if k < save_sample_paths:
            risk.to_csv(os.path.join(sample_dir, f"path_{k}_risk_policy.csv"), index=False)
            daily.to_csv(os.path.join(sample_dir, f"path_{k}_vault_daily.csv"), index=False)

    summary = pd.DataFrame(stats_rows)
    summary.to_csv(os.path.join(sdir, "summary.csv"), index=False)

    histogram(np.array(max_liqs), "Max Daily Liquidations (per path)", "max daily liquidations",
              os.path.join(sdir, "hist_max_daily_liquidations.png"))
    histogram(np.array(max_liq_vols), "Max Daily Liquidation Volume (per path)", "max daily liq volume",
              os.path.join(sdir, "hist_max_daily_liq_volume.png"))
    histogram(np.array(max_oas), "Max OAS (bps) (per path)", "max OAS (bps)",
              os.path.join(sdir, "hist_max_oas.png"))
    histogram(np.array(min_liq), "Min Liquidity (per path)", "min liquidity",
              os.path.join(sdir, "hist_min_liquidity.png"))
    histogram(np.array(max_dd), "Max Drawdown (per path)", "max drawdown (negative)",
              os.path.join(sdir, "hist_max_drawdown.png"))
    histogram(np.array(emergency_days), "Emergency Days (per path)", "days in EMERGENCY_UNWIND",
              os.path.join(sdir, "hist_emergency_days.png"))

    risk_surface_plot(p, outdir)
    print(f"[stochastic] done. outputs in {sdir}/ and policy surfaces in {outdir}/")


# =========================
# 12) CLI / Main (parse_known_args for Jupyter)
# =========================

def parse_args():
    ap = argparse.ArgumentParser(add_help=True)
    ap.add_argument("--mode", choices=["scenarios", "stochastic", "both"], default="both")
    ap.add_argument("--outdir", default="phase0_outputs_v03_4")
    ap.add_argument("--paths", type=int, default=None)
    ap.add_argument("--save-sample-paths", type=int, default=5)
    args, _unknown = ap.parse_known_args()  # ✅ ignore ipykernel -f ...
    return args

def main():
    args = parse_args()
    outdir = args.outdir
    ensure_dir(outdir)

    np.random.seed(PARAMS["seed"])
    random.seed(PARAMS["seed"])

    print(args.mode)

    if args.mode in ("scenarios", "both"):
        run_scenarios(PARAMS, outdir)

    if args.mode in ("stochastic", "both"):
        n_paths = args.paths if args.paths is not None else PARAMS["stochastic"]["n_paths_default"]
        run_stochastic(PARAMS, outdir, n_paths=n_paths, save_sample_paths=args.save_sample_paths)

    print("\nAll done.")
    print(f"Outputs at: {outdir}/")

if __name__ == "__main__":
    main()
